# Comparing all three physically-motivated feature approaches, plus SMARD

This project sits alongside two other ways of turning weather + MaStR
capacity into a wind-onshore capacity factor, all developed across
`mastr-power-capacities-germany`, `delu-headline-forecast`, and this
project. Laid out from least to most physically detailed:

1. **PECD's own PEON zone capacity factor x MaStR zone capacity**
   (`~/research/delu-headline-forecast/dhf/pecd.py`) -- trusts PECD's
   official numbers wholesale, just reweights them by MaStR's installed
   capacity. No bias adjustment or per-turbine detail *by construction* --
   it's already PECD's finished product.
2. **Physics-based reconstruction from raw ERA5 + MaStR**
   (`~/research/delu-headline-forecast/dhf/physics.py`) -- computed from
   scratch, but with the simplest possible physics: one generic turbine
   power curve for the entire fleet, no GWA2 bias correction.
3. **This project's PECD replication** -- computed from scratch with
   PECD's actual documented methodology as closely as achievable: GWA2
   bias-adjusted wind speed, alpha-based hub-height extrapolation,
   per-plant power curves (real matched curve or specific-power-based
   generic fallback).

Plus **SMARD's actual DE-LU wind-onshore generation** as the real-world
ground truth none of the three are derived from.

All four series now use a fleet-vintage-correct weighting: approaches 1,
2, and the SMARD-implied CF use MaStR's *time-varying* monthly capacity
as the weighting denominator (`pecdr.capacity_weighting`); approach 3
(this project) runs each year's weather through **that year's own**
start-of-year fleet (`pipeline/08_build_wind_onshore_plant_panel.py`,
`pipeline/10_compute_wind_onshore_full_year.py`) rather than today's
fleet applied retroactively.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pecdr.capacity_weighting import capacity_weighted_cf, total_hourly_capacity
from pecdr.domain import FULL_YEAR_BACKTEST_YEARS
from pecdr.paths import ProjPaths

paths = ProjPaths()
paths.ensure_directories()

PERIOD_LABEL = f"{FULL_YEAR_BACKTEST_YEARS[0]}-{FULL_YEAR_BACKTEST_YEARS[-1]}"

## Load all four series, all as capacity factor, restricted to the common backtest period

In [2]:
mastr_peon_month = pd.read_parquet(paths.mastr_capacity_by_peon_month_file)

# Approach 1: PECD's own PEON zone CF x time-varying MaStR PEON capacity
pecd_peon_cf = pd.read_parquet(paths.pecd_wind_onshore_capacity_factors_file)
approach1 = capacity_weighted_cf(pecd_peon_cf, mastr_peon_month)

# Approach 2: delu's own raw-ERA5 + generic-curve PEON zone CF, same weights
delu_own_cf = pd.read_parquet(paths.delu_own_wind_onshore_capacity_factors_file)
delu_own_cf.index.name = "timestamp"
approach2 = capacity_weighted_cf(delu_own_cf, mastr_peon_month)

# Approach 3: this project's replication (already national, each year's own fleet)
approach3 = pd.read_parquet(paths.wind_onshore_full_year_cf_file)["national"]

# SMARD actual generation (MW) -> implied CF via time-varying total capacity
target_panel = pd.read_parquet(paths.target_panel_file)
total_capacity_hourly = total_hourly_capacity(mastr_peon_month, target_panel.index)
smard_implied_cf = target_panel["wind_onshore"] / total_capacity_hourly

comparison = pd.DataFrame({
    "1_pecd_official_x_mastr": approach1,
    "2_delu_physics_no_bias": approach2,
    "3_pecd_replication": approach3,
    "4_smard_implied": smard_implied_cf,
})
comparison = comparison.loc[FULL_YEAR_BACKTEST_YEARS[0]:FULL_YEAR_BACKTEST_YEARS[-1]].dropna(how="all")
print(f"Hours with at least one series: {len(comparison):,}")
comparison.describe()

Hours with at least one series: 61,368


,1_pecd_official_x_mastr,2_delu_physics_no_bias,3_pecd_replication,4_smard_implied
count,61368.000000,61368.000000,61368.000000,61368.000000
mean,0.225832,0.134739,0.268058,0.208576
std,0.194603,0.191162,0.229566,0.165695
min,0.000058,0.000000,0.000071,0.000749
25%,0.077631,0.016175,0.084652,0.078996
50%,0.160889,0.054510,0.191672,0.160587
75%,0.321965,0.166328,0.401208,0.299478
max,0.845861,0.997872,0.901726,0.803940


## Full-period time series, all four series

In [3]:
fig, ax = plt.subplots(figsize=(14, 5))
for col, label in [
    ("1_pecd_official_x_mastr", "1. PECD official x MaStR"),
    ("2_delu_physics_no_bias", "2. delu physics (no bias, generic curve)"),
    ("3_pecd_replication", "3. This project's PECD replication"),
    ("4_smard_implied", "4. SMARD actual (implied CF)"),
]:
    comparison[col].plot(ax=ax, label=label, alpha=0.7, linewidth=0.3)
ax.set_ylabel("Germany-wide onshore wind capacity factor")
ax.set_title(f"All four approaches, {PERIOD_LABEL}")
ax.legend()
fig.tight_layout()
fig.savefig(paths.images_path / "12_all_approaches_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

/tmp/ipykernel_468515/3261441069.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


```{figure} ../../output/images/12_all_approaches_timeseries.png
:name: fig-12-all-approaches-timeseries
All four wind-onshore capacity factor series, 2019-2025.
```

## Each modeled approach vs. SMARD's actual (implied) capacity factor

SMARD is the one series here nobody derived their number from -- the
genuine external ground truth, unlike approach 1 vs. 3 (which both trace
back to PECD in different ways) or approach 2 vs. 3 (both derived from the
same raw ERA5).

In [4]:
stats_rows = []
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, label in zip(axes, ["1_pecd_official_x_mastr", "2_delu_physics_no_bias", "3_pecd_replication"], ["1. PECD official x MaStR", "2. delu physics", "3. PECD replication"]):
    sub = comparison[[col, "4_smard_implied"]].dropna()
    diff = sub[col] - sub["4_smard_implied"]
    corr = sub[col].corr(sub["4_smard_implied"])
    stats_rows.append({"approach": label, "n_hours": len(sub), "mae": float(diff.abs().mean()), "bias": float(diff.mean()), "corr": float(corr)})

    ax.scatter(sub["4_smard_implied"], sub[col], s=2, alpha=0.05)
    lims = [0, max(sub.max())]
    ax.plot(lims, lims, "k--", alpha=0.5, label="1:1")
    ax.set_xlabel("SMARD-implied CF")
    ax.set_ylabel(f"{label} CF")
    ax.set_title(f"{label}\nMAE={diff.abs().mean():.3f}, corr={corr:.3f}")
fig.tight_layout()
fig.savefig(paths.images_path / "12_approaches_vs_smard.png", dpi=150, bbox_inches="tight")
plt.show()

stats = pd.DataFrame(stats_rows)
stats

/tmp/ipykernel_468515/454666041.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,approach,n_hours,mae,bias,corr
0,1. PECD official x MaStR,61368,0.028846,0.017256,0.985480
1,2. delu physics,61368,0.089275,-0.073837,0.930635
2,3. PECD replication,61368,0.064821,0.059483,0.982652


```{figure} ../../output/images/12_approaches_vs_smard.png
:name: fig-12-approaches-vs-smard
Each modeled approach vs. SMARD's actual (implied) capacity factor, 2019-2025.
```

## Does each approach's fit to SMARD hold up year to year?

The pooled stats above could hide a single unusual year. Breaking out
bias/MAE/corr by year checks whether the fit is stable across the whole
backtest period, not just true on average.

In [5]:
yearly_rows = []
for col, label in [
    ("1_pecd_official_x_mastr", "1. PECD official x MaStR"),
    ("2_delu_physics_no_bias", "2. delu physics"),
    ("3_pecd_replication", "3. PECD replication"),
]:
    sub = comparison[[col, "4_smard_implied"]].dropna()
    diff = sub[col] - sub["4_smard_implied"]
    for year, group in diff.groupby(diff.index.year):
        approach_group = sub.loc[group.index, col]
        smard_group = sub.loc[group.index, "4_smard_implied"]
        yearly_rows.append({
            "approach": label,
            "year": year,
            "n_hours": len(group),
            "mae": float(group.abs().mean()),
            "bias": float(group.mean()),
            "corr": float(approach_group.corr(smard_group)),
        })

yearly_stats = pd.DataFrame(yearly_rows).pivot(index="year", columns="approach", values=["bias", "mae", "corr"])
yearly_stats

bias                                      \
approach 1. PECD official x MaStR 2. delu physics 3. PECD replication   
year                                                                    
2019                     0.015407       -0.077621            0.049620   
2020                     0.016512       -0.076610            0.053066   
2021                     0.014508       -0.073358            0.048283   
2022                     0.016192       -0.070899            0.056207   
2023                     0.022773       -0.068266            0.071542   
2024                     0.020337       -0.071299            0.072149   
2025                     0.015059       -0.078805            0.065493   

                              mae                                      \
approach 1. PECD official x MaStR 2. delu physics 3. PECD replication   
year                                                                    
2019                     0.028326        0.092008            0.056970   
2020                     0.029382        0.092651            0.059797   
2021                     0.025943        0.083783            0.053862   
2022                     0.028378        0.089511            0.061981   
2023                     0.032821        0.091135            0.075487   
2024                     0.030099        0.086221            0.076046   
2025                     0.026970        0.089612            0.069590   

                             corr                                      
approach 1. PECD official x MaStR 2. delu physics 3. PECD replication  
year                                                                   
2019                     0.989379        0.942464            0.987597  
2020                     0.987666        0.930433            0.986082  
2021                     0.985472        0.934322            0.984449  
2022                     0.985751        0.929146            0.984112  
2023                     0.984133        0.932961            0.982766  
2024                     0.984614        0.931307            0.981191  
2025                     0.979786        0.905886            0.981601